# 08_paper / tables.ipynb

**Purpose:** Consolidate all paper-quality tables. Produces:

1. **Final consolidated tables** (curated, ready for paper) → `finalized_outputs/tables/`
2. **LaTeX versions** of the headline tables (booktabs format) → `finalized_outputs/latex/`
3. **A master index** of every table with description, source, and which paper section uses it.

**Inputs:** all CSVs in `paper/tables/` (already produced by streams 01–05 and notebooks 07_eval).

**Outputs:**
- `finalized_outputs/tables/` — clean, paper-ready CSVs (renamed for clarity)
- `finalized_outputs/latex/*.tex` — LaTeX booktabs source for each headline table
- `finalized_outputs/tables/INDEX.md` — describes every table

**No model is loaded or trained here. Reads CSVs only — runs in seconds.**


In [1]:
# === Imports & path setup ===
from __future__ import annotations
import sys, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import PAPER, TABLES, FIGURES

FINAL = ROOT / "finalized_outputs"
FINAL_TABLES = FINAL / "tables"
FINAL_LATEX  = FINAL / "latex"
FINAL_TABLES.mkdir(parents=True, exist_ok=True)
FINAL_LATEX.mkdir(parents=True, exist_ok=True)

print("Repo root      :", ROOT)
print("paper/tables   :", TABLES, "  exists:", TABLES.exists())
print("Final tables   :", FINAL_TABLES)
print("Final latex    :", FINAL_LATEX)


Repo root      : e:\eve-rag
paper/tables   : E:\eve-rag\paper\tables   exists: True
Final tables   : e:\eve-rag\finalized_outputs\tables
Final latex    : e:\eve-rag\finalized_outputs\latex


## 1. Inventory all source tables

In [2]:
# List every CSV currently in paper/tables/
all_csvs = sorted(TABLES.glob("*.csv"))
inventory = []
for p in all_csvs:
    try:
        df = pd.read_csv(p)
        inventory.append({"file": p.name, "rows": len(df), "cols": len(df.columns),
                          "columns": ", ".join(df.columns[:6]) + ("…" if len(df.columns) > 6 else "")})
    except Exception as e:
        inventory.append({"file": p.name, "rows": -1, "cols": -1, "columns": f"[error: {e}]"})

inv_df = pd.DataFrame(inventory)
print(f"Found {len(inv_df)} CSVs in paper/tables/")
inv_df


Found 32 CSVs in paper/tables/


,file,rows,cols,columns
0,ablation_results.csv,25,7,"ablation, dataset, thr, acc, f1, f1_macro…"
1,ablations_delta_vs_full.csv,12,8,"model, dataset, Δauroc, Δauprc, Δaccuracy, Δf1…"
2,ablations_full.csv,15,10,"model, dataset, n, auroc, auprc, accuracy…"
3,all_streams_and_baselines_summary.csv,42,11,"method, dataset, acc, precision, recall, f1…"
4,baseline_results.csv,16,7,"method, dataset, acc, precision, recall, f1…"
5,caption_length_stats.csv,3,6,"dataset, rows, mean_chars, median_chars, mean_..."
6,clip_baseline_results.csv,8,7,"method, dataset, acc, precision, recall, f1…"
7,cross_dataset_eval_combined.csv,14,10,"dataset, model, n, auroc, auprc, accuracy…"
8,cross_dataset_eval_mmfakebench.csv,8,9,"model, n, auroc, auprc, accuracy, f1…"
9,cross_dataset_eval_verite.csv,6,9,"model, n, auroc, auprc, accuracy, f1…"


## 2. Helper: LaTeX booktabs writer

In [3]:
def df_to_latex_booktabs(df: pd.DataFrame,
                         caption: str,
                         label: str,
                         column_format: str | None = None,
                         float_fmt: str = "{:.4f}".format,
                         out_path: Path | None = None) -> str:
    """Render a DataFrame as a paper-ready LaTeX table using booktabs."""
    df = df.copy()
    # Cast numeric cols
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].map(lambda x: float_fmt(x) if pd.notna(x) else "")
        elif pd.api.types.is_integer_dtype(df[c]):
            df[c] = df[c].map(lambda x: f"{int(x):,}" if pd.notna(x) else "")

    n_cols = len(df.columns)
    if column_format is None:
        column_format = "l" + "r" * (n_cols - 1)

    # Build body
    header = " & ".join([f"\\textbf{{{c}}}" for c in df.columns]) + " \\\\"
    body_rows = []
    for _, row in df.iterrows():
        body_rows.append(" & ".join(str(v) for v in row.values) + " \\\\")
    body = "\n".join(body_rows)

    latex = (
        "\\begin{table}[!t]\n"
        "\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        f"\\begin{{tabular}}{{{column_format}}}\n"
        "\\toprule\n"
        f"{header}\n"
        "\\midrule\n"
        f"{body}\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\end{table}\n"
    )
    if out_path is not None:
        Path(out_path).write_text(latex, encoding="utf-8")
        print(f"  Wrote {out_path}")
    return latex


## 3. Headline Table 1 — Cross-dataset evaluation (the paper's hero table)

In [4]:
# This is the most important table for the paper. GSAF + 4 ablations + per-stream baselines
# on all 3 test sets, AUROC + F1 + accuracy.
src = TABLES / "cross_dataset_eval_combined.csv"
df = pd.read_csv(src)

# Pretty model names + ordering
RENAME_MODEL = {
    "GSAF (full)":            "GSAF (ours)",
    "GSAF_default":           "GSAF (ours)",
    "GSAF_no_a":              "GSAF − Stream A",
    "GSAF_no_b":              "GSAF − Stream B",
    "GSAF_no_c":              "GSAF − Stream C",
    "GSAF_mlp_fusion":        "GSAF (MLP fusion)",
    "Stream A (entities)":    "Stream A (entities)",
    "Stream B (CLIP)":        "Stream B (CLIP)",
    "Stream C (NLI)":         "Stream C (NLI)",
}
ORDER = [
    "Stream A (entities)", "Stream B (CLIP)", "Stream C (NLI)",
    "GSAF − Stream A", "GSAF − Stream B", "GSAF − Stream C",
    "GSAF (MLP fusion)", "GSAF (ours)",
]

# Normalize cross_dataset rows (MMFB + VERITE only)
df["model"]   = df["model"].map(lambda x: RENAME_MODEL.get(x, x))
df["dataset"] = (df["dataset"]
                 .str.replace("-test", "", regex=False)
                 .str.replace("MMFakeBench", "MMFB", regex=False))
cross_part = df[["model", "dataset", "auroc", "f1"]].copy()

# Pull NewsCLIPpings rows from newsclippings_eval.csv and add a 'dataset' col
nc_path = TABLES / "newsclippings_eval.csv"
if nc_path.exists():
    nc = pd.read_csv(nc_path)
    nc["model"] = nc["model"].map(lambda x: RENAME_MODEL.get(x, x))
    nc["dataset"] = "NewsCLIPpings"
    nc_part = nc[["model", "dataset", "auroc", "f1"]].copy()
    full_long = pd.concat([nc_part, cross_part], ignore_index=True)
else:
    full_long = cross_part

# Pivot: rows = model, cols = (metric, dataset)
pivot = full_long.pivot_table(
    index="model",
    columns="dataset",
    values=["auroc", "f1"],
    aggfunc="first",
)

# Reorder index AND columns (datasets in NC → MMFB → VERITE order)
pivot = pivot.reindex([m for m in ORDER if m in pivot.index])
DATASET_ORDER = ["NewsCLIPpings", "MMFB", "VERITE"]
present = [d for d in DATASET_ORDER if d in pivot.columns.get_level_values(1)]
pivot = pivot.reindex(columns=pd.MultiIndex.from_product([["auroc", "f1"], present]))

# Flatten cols: ('auroc','MMFB') → 'MMFB AUROC'
pivot.columns = [f"{ds} {m.upper()}" for m, ds in pivot.columns]
pivot = pivot.round(4)

print("Headline cross-dataset table:")
display(pivot)

# Save
out_csv = FINAL_TABLES / "01_headline_cross_dataset.csv"
pivot.reset_index().to_csv(out_csv, index=False, float_format="%.4f")
print(f"\nSaved {out_csv}")

# LaTeX
df_to_latex_booktabs(
    pivot.reset_index(),
    caption="Cross-dataset out-of-distribution evaluation. AUROC and F1 on three benchmarks. The full GSAF model attains the strongest in-distribution performance (NewsCLIPpings) and substantially outperforms ablations on MMFakeBench, validating the per-sample gating mechanism.",
    label="tab:cross_dataset",
    out_path=FINAL_LATEX / "01_headline_cross_dataset.tex",
)

Headline cross-dataset table:


,NewsCLIPpings AUROC,MMFB AUROC,VERITE AUROC,NewsCLIPpings F1,MMFB F1,VERITE F1
model,,,,,,
Stream A (entities),NaN,0.6541,NaN,NaN,0.7070,NaN
Stream B (CLIP),0.7391,0.6335,NaN,0.6411,0.4484,NaN
Stream C (NLI),0.5181,0.6943,0.4594,0.4918,0.1459,0.1776
GSAF − Stream A,0.7523,0.6524,0.4737,0.6666,0.6687,0.7761
GSAF − Stream B,0.5249,0.6527,0.4896,0.5348,0.7077,0.7024
GSAF − Stream C,0.7547,0.6287,0.4813,0.6630,0.7209,0.7930
GSAF (MLP fusion),0.7626,0.6596,0.4854,0.6866,0.7296,0.7922
GSAF (ours),0.7632,0.8783,0.4813,0.6688,0.8428,0.7633



Saved e:\eve-rag\finalized_outputs\tables\01_headline_cross_dataset.csv
  Wrote e:\eve-rag\finalized_outputs\latex\01_headline_cross_dataset.tex


'\\begin{table}[!t]\n\\centering\n\\caption{Cross-dataset out-of-distribution evaluation. AUROC and F1 on three benchmarks. The full GSAF model attains the strongest in-distribution performance (NewsCLIPpings) and substantially outperforms ablations on MMFakeBench, validating the per-sample gating mechanism.}\n\\label{tab:cross_dataset}\n\\begin{tabular}{lrrrrrr}\n\\toprule\n\\textbf{model} & \\textbf{NewsCLIPpings AUROC} & \\textbf{MMFB AUROC} & \\textbf{VERITE AUROC} & \\textbf{NewsCLIPpings F1} & \\textbf{MMFB F1} & \\textbf{VERITE F1} \\\\\n\\midrule\nStream A (entities) &  & 0.6541 &  &  & 0.7070 &  \\\\\nStream B (CLIP) & 0.7391 & 0.6335 &  & 0.6411 & 0.4484 &  \\\\\nStream C (NLI) & 0.5181 & 0.6943 & 0.4594 & 0.4918 & 0.1459 & 0.1776 \\\\\nGSAF − Stream A & 0.7523 & 0.6524 & 0.4737 & 0.6666 & 0.6687 & 0.7761 \\\\\nGSAF − Stream B & 0.5249 & 0.6527 & 0.4896 & 0.5348 & 0.7077 & 0.7024 \\\\\nGSAF − Stream C & 0.7547 & 0.6287 & 0.4813 & 0.6630 & 0.7209 & 0.7930 \\\\\nGSAF (MLP fusio

## 4. Headline Table 2 — Ablation deltas (Δ vs full GSAF)

In [5]:
src = TABLES / "ablations_delta_vs_full.csv"
df = pd.read_csv(src)
print("Source columns:", df.columns.tolist())

df["model"]   = df["model"].map(lambda x: RENAME_MODEL.get(x, x))
df["dataset"] = df["dataset"].str.replace("_test", "").str.replace("mmfakebench", "MMFB")\
                                .str.replace("newsclippings", "NC").str.replace("verite", "VERITE")

pivot = df.pivot_table(index="model", columns="dataset", values="Δauroc", aggfunc="first")
pivot = pivot.round(4)
pivot.columns = [f"Δ AUROC ({c})" for c in pivot.columns]

# Reorder
preferred = ["GSAF − Stream A", "GSAF − Stream B", "GSAF − Stream C", "GSAF (MLP fusion)"]
pivot = pivot.reindex([m for m in preferred if m in pivot.index])

print("Ablation Δ-AUROC table:")
display(pivot)

out_csv = FINAL_TABLES / "02_ablation_delta_auroc.csv"
pivot.reset_index().to_csv(out_csv, index=False, float_format="%+.4f")
print(f"\nSaved {out_csv}")

df_to_latex_booktabs(
    pivot.reset_index(),
    caption="Ablation analysis: change in AUROC when removing each component or replacing the gating mechanism with a flat MLP. Negative values indicate the full GSAF model outperforms the ablation. Stream B is the most critical channel; the gating mechanism contributes ${\sim}22$ AUROC points on MMFakeBench.",
    label="tab:ablation_delta",
    float_fmt="{:+.4f}".format,
    out_path=FINAL_LATEX / "02_ablation_delta_auroc.tex",
)


Source columns: ['model', 'dataset', 'Δauroc', 'Δauprc', 'Δaccuracy', 'Δf1', 'Δprecision', 'Δrecall']
Ablation Δ-AUROC table:


<>:26: SyntaxWarning: invalid escape sequence '\s'
<>:26: SyntaxWarning: invalid escape sequence '\s'
C:\Users\User\AppData\Local\Temp\ipykernel_23616\2401892576.py:26: SyntaxWarning: invalid escape sequence '\s'
  caption="Ablation analysis: change in AUROC when removing each component or replacing the gating mechanism with a flat MLP. Negative values indicate the full GSAF model outperforms the ablation. Stream B is the most critical channel; the gating mechanism contributes ${\sim}22$ AUROC points on MMFakeBench.",


,Δ AUROC (MMFB),Δ AUROC (NC),Δ AUROC (VERITE)
model,,,
GSAF − Stream A,-0.2259,-0.0108,-0.0076
GSAF − Stream B,-0.2256,-0.2382,0.0084
GSAF − Stream C,-0.2496,-0.0084,-0.0000
GSAF (MLP fusion),-0.2187,-0.0006,0.0041



Saved e:\eve-rag\finalized_outputs\tables\02_ablation_delta_auroc.csv
  Wrote e:\eve-rag\finalized_outputs\latex\02_ablation_delta_auroc.tex


'\\begin{table}[!t]\n\\centering\n\\caption{Ablation analysis: change in AUROC when removing each component or replacing the gating mechanism with a flat MLP. Negative values indicate the full GSAF model outperforms the ablation. Stream B is the most critical channel; the gating mechanism contributes ${\\sim}22$ AUROC points on MMFakeBench.}\n\\label{tab:ablation_delta}\n\\begin{tabular}{lrrr}\n\\toprule\n\\textbf{model} & \\textbf{Δ AUROC (MMFB)} & \\textbf{Δ AUROC (NC)} & \\textbf{Δ AUROC (VERITE)} \\\\\n\\midrule\nGSAF − Stream A & -0.2259 & -0.0108 & -0.0076 \\\\\nGSAF − Stream B & -0.2256 & -0.2382 & +0.0084 \\\\\nGSAF − Stream C & -0.2496 & -0.0084 & -0.0000 \\\\\nGSAF (MLP fusion) & -0.2187 & -0.0006 & +0.0041 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}\n'

## 5. Headline Table 3 — Dataset summary

In [6]:
src = TABLES / "dataset_summary.csv"
if src.exists():
    df = pd.read_csv(src)
    print("Source dataset_summary.csv:")
    display(df)
    out_csv = FINAL_TABLES / "03_dataset_summary.csv"
    df.to_csv(out_csv, index=False)
    print(f"Saved {out_csv}")
    df_to_latex_booktabs(
        df,
        caption="Datasets used in this work. NewsCLIPpings provides the primary in-distribution training and evaluation; MMFakeBench and VERITE are held-out for cross-dataset testing.",
        label="tab:datasets",
        float_fmt="{:,.0f}".format,
        out_path=FINAL_LATEX / "03_dataset_summary.tex",
    )
else:
    print("dataset_summary.csv not found, skipping.")


Source dataset_summary.csv:


,dataset_split,rows,label_0_match,label_1_mismatch,has_image_rows,unique_image_ids,empty_captions
0,NewsCLIPpings train,71072,35536,35536,0,59876,0
1,NewsCLIPpings val,7024,3512,3512,0,5943,0
2,NewsCLIPpings test,7264,3632,3632,0,6147,0
3,VERITE test,1001,338,663,621,495,0
4,MMFakeBench val,1000,300,700,1000,1000,0
5,MMFakeBench test,10000,3000,7000,10000,10000,0
6,All,97361,46318,51043,11621,83461,0


Saved e:\eve-rag\finalized_outputs\tables\03_dataset_summary.csv
  Wrote e:\eve-rag\finalized_outputs\latex\03_dataset_summary.tex


## 6. Headline Table 4 — Per-stream individual results

In [7]:
# Reads each stream's results CSV and merges into a single per-dataset table.
rows = []
for stream, src in [("Stream A", "stream_a_results.csv"),
                    ("Stream B", "stream_b_results.csv"),
                    ("Stream C", "stream_c_results.csv")]:
    p = TABLES / src
    if not p.exists():
        print(f"Skipping {src} (not found).")
        continue
    sdf = pd.read_csv(p)
    print(f"\n{stream} columns:", sdf.columns.tolist())
    # Heuristic: pick the rows that look like test splits
    if "split" in sdf.columns and "dataset" in sdf.columns:
        sdf = sdf[sdf["split"].astype(str).str.contains("test", case=False, na=False)].copy()
    sdf["stream"] = stream
    rows.append(sdf)

if rows:
    streams_df = pd.concat(rows, ignore_index=True, sort=False)
    # Normalise columns
    keep = [c for c in ["stream","dataset","split","tuning","thr","acc","f1","f1_macro","auroc"] if c in streams_df.columns]
    streams_df = streams_df[keep]
    print("\nMerged per-stream results:")
    display(streams_df)
    out_csv = FINAL_TABLES / "04_per_stream_results.csv"
    streams_df.to_csv(out_csv, index=False)
    print(f"Saved {out_csv}")



Stream A columns: ['method', 'dataset', 'acc', 'precision', 'recall', 'f1', 'auroc']

Stream B columns: ['dataset', 'acc', 'precision', 'recall', 'f1', 'auroc']

Stream C columns: ['dataset', 'tuning', 'threshold', 'acc', 'precision', 'recall', 'f1', 'macro_f1', 'auroc']

Merged per-stream results:


,stream,dataset,tuning,acc,f1,auroc
0,Stream A,mmfakebench_test,NaN,62.56,70.70,65.41
1,Stream A,mmfakebench_test,NaN,70.00,82.35,NaN
2,Stream A,mmfakebench_test,NaN,58.10,65.08,64.14
3,Stream A,verite_test_local,NaN,53.30,60.60,53.83
4,Stream A,verite_test_local,NaN,66.51,79.88,NaN
5,Stream A,verite_test_local,NaN,48.47,55.80,47.82
6,Stream B,newsclippings_val,NaN,66.22,64.48,73.42
7,Stream B,newsclippings_test,NaN,65.87,64.11,73.91
8,Stream B,mmfakebench_test,NaN,46.95,44.84,63.35
9,Stream B,verite_test_local,NaN,63.77,77.25,49.12


Saved e:\eve-rag\finalized_outputs\tables\04_per_stream_results.csv


## 7. Headline Table 5 — Per-class breakdown of GSAF on NewsCLIPpings

In [8]:
src = TABLES / "newsclippings_eval_per_class.csv"
if src.exists():
    df = pd.read_csv(src)
    df.columns = ["class"] + list(df.columns[1:])
    print("Per-class report (GSAF, NewsCLIPpings test):")
    display(df)
    out_csv = FINAL_TABLES / "05_per_class_newsclippings.csv"
    df.to_csv(out_csv, index=False, float_format="%.4f")
    print(f"Saved {out_csv}")
    df_to_latex_booktabs(
        df,
        caption="Per-class precision, recall and F1 of the full GSAF model on NewsCLIPpings test.",
        label="tab:per_class_nc",
        out_path=FINAL_LATEX / "05_per_class_newsclippings.tex",
    )


Per-class report (GSAF, NewsCLIPpings test):


,class,precision,recall,f1-score,support
0,genuine (0),0.6685,0.7412,0.7030,3632.0000
1,OOC (1),0.7096,0.6324,0.6688,3632.0000
2,accuracy,0.6868,0.6868,0.6868,0.6868
3,macro avg,0.6890,0.6868,0.6859,7264.0000
4,weighted avg,0.6890,0.6868,0.6859,7264.0000


Saved e:\eve-rag\finalized_outputs\tables\05_per_class_newsclippings.csv
  Wrote e:\eve-rag\finalized_outputs\latex\05_per_class_newsclippings.tex


## 8. Copy supporting tables verbatim (no transformation)

In [9]:
# These tables are useful but don't need restyling; copy as-is to finalized_outputs.
SUPPORTING = [
    ("ablations_full.csv",                  "06_ablations_full.csv"),
    ("ablation_results.csv",                "07_ablation_legacy.csv"),
    ("fusion_results.csv",                  "08_fusion_results_full.csv"),
    ("newsclippings_eval.csv",              "09_newsclippings_eval_all_models.csv"),
    ("cross_dataset_eval_mmfakebench.csv",  "10_cross_dataset_mmfakebench.csv"),
    ("cross_dataset_eval_verite.csv",       "11_cross_dataset_verite.csv"),
    ("qualitative_examples.csv",            "12_qualitative_examples.csv"),
    ("qualitative_gate_attribution.csv",    "13_qualitative_gate_attribution.csv"),
    ("baseline_results.csv",                "14_baseline_results.csv"),
    ("clip_baseline_results.csv",           "15_clip_baseline_results.csv"),
]

for src_name, dst_name in SUPPORTING:
    src = TABLES / src_name
    dst = FINAL_TABLES / dst_name
    if src.exists():
        shutil.copy2(src, dst)
        print(f"  copied {src_name} → {dst.name}")
    else:
        print(f"  [skip] {src_name} not found")


  copied ablations_full.csv → 06_ablations_full.csv
  copied ablation_results.csv → 07_ablation_legacy.csv
  copied fusion_results.csv → 08_fusion_results_full.csv
  copied newsclippings_eval.csv → 09_newsclippings_eval_all_models.csv
  copied cross_dataset_eval_mmfakebench.csv → 10_cross_dataset_mmfakebench.csv
  copied cross_dataset_eval_verite.csv → 11_cross_dataset_verite.csv
  copied qualitative_examples.csv → 12_qualitative_examples.csv
  copied qualitative_gate_attribution.csv → 13_qualitative_gate_attribution.csv
  copied baseline_results.csv → 14_baseline_results.csv
  copied clip_baseline_results.csv → 15_clip_baseline_results.csv


## 9. Write master INDEX.md

In [10]:
INDEX = '''# Finalized tables — INDEX

This folder is the **single source of truth** for every table that goes into the paper, slides, or appendix.

| # | File | Purpose | Used in |
|---|---|---|---|
| 01 | `01_headline_cross_dataset.csv` | The hero table: GSAF + ablations + streams on all 3 test sets, AUROC + F1 | Main results §5 |
| 02 | `02_ablation_delta_auroc.csv`   | Δ-AUROC vs full GSAF for each ablation, per dataset                       | Ablation §5.x |
| 03 | `03_dataset_summary.csv`        | NewsCLIPpings / MMFakeBench / VERITE row counts, splits, label balance     | Datasets §3 |
| 04 | `04_per_stream_results.csv`     | Each stream's standalone metrics on every test split                       | Method §4 |
| 05 | `05_per_class_newsclippings.csv`| Per-class precision/recall/F1 of GSAF on NewsCLIPpings test                | Main results §5 |
| 06 | `06_ablations_full.csv`         | Full ablation grid (model × dataset × all metrics)                         | Appendix |
| 07 | `07_ablation_legacy.csv`        | Original ablation table from `fusion_training.ipynb` (sanity)              | Appendix |
| 08 | `08_fusion_results_full.csv`    | All splits, both default + tuned thresholds                                | Appendix |
| 09 | `09_newsclippings_eval_all_models.csv` | NewsCLIPpings eval of every variant + streams                       | Appendix |
| 10 | `10_cross_dataset_mmfakebench.csv` | MMFakeBench-only OOD table                                              | Appendix |
| 11 | `11_cross_dataset_verite.csv`     | VERITE-only OOD table                                                  | Appendix |
| 12 | `12_qualitative_examples.csv`     | The 24 illustrative pairs picked for the qualitative figures           | Appendix |
| 13 | `13_qualitative_gate_attribution.csv` | Average gate weights per (dataset, bucket)                          | Discussion §6 |
| 14 | `14_baseline_results.csv`         | Naïve text-only and image-only baselines                                | Main results §5 |
| 15 | `15_clip_baseline_results.csv`    | Vanilla CLIP-similarity baseline                                        | Main results §5 |

LaTeX (booktabs) sources for the headline tables are in `../latex/`.
'''
(FINAL_TABLES / "INDEX.md").write_text(INDEX, encoding="utf-8")
print("Wrote INDEX.md")


Wrote INDEX.md


## 10. Final summary

In [11]:
print("="*70)
print(" FINAL TABLES COMPLETE")
print("="*70)
print(f"  finalized_outputs/tables/  : {len(list(FINAL_TABLES.glob('*.csv')))} CSVs + INDEX.md")
print(f"  finalized_outputs/latex/   : {len(list(FINAL_LATEX.glob('*.tex')))} .tex files")
print()
print("Files in finalized_outputs/tables/:")
for p in sorted(FINAL_TABLES.iterdir()):
    print(f"  {p.name}")
print()
print("Next: 08_paper/figures.ipynb")


 FINAL TABLES COMPLETE
  finalized_outputs/tables/  : 15 CSVs + INDEX.md
  finalized_outputs/latex/   : 4 .tex files

Files in finalized_outputs/tables/:
  01_headline_cross_dataset.csv
  02_ablation_delta_auroc.csv
  03_dataset_summary.csv
  04_per_stream_results.csv
  05_per_class_newsclippings.csv
  06_ablations_full.csv
  07_ablation_legacy.csv
  08_fusion_results_full.csv
  09_newsclippings_eval_all_models.csv
  10_cross_dataset_mmfakebench.csv
  11_cross_dataset_verite.csv
  12_qualitative_examples.csv
  13_qualitative_gate_attribution.csv
  14_baseline_results.csv
  15_clip_baseline_results.csv
  INDEX.md

Next: 08_paper/figures.ipynb
